In [1]:
import os
import json
import glob

In [2]:
JSON_DIR = "B_PART_LIGHT"

json_files = glob.glob(
    os.path.join(JSON_DIR, "*.json")
)

In [3]:
def classify_event(summary):

    person = summary.get("person", 0)

    bicycle = summary.get("bicycle", 0)

    motorcycle = summary.get("motorcycle", 0)

    two_wheel = bicycle + motorcycle

    # -----------------------
    # 규칙 1
    # -----------------------

    if two_wheel == 0:

        return "기타"

    # -----------------------
    # 규칙 2
    # -----------------------

    if bicycle >= motorcycle * 2:

        return "자전거 인도 주행"

    # -----------------------
    # 규칙 3
    # -----------------------

    if motorcycle >= 20:

        return "이륜 이동수단 헬멧 미착용 후보"

    # -----------------------
    # 규칙 4
    # -----------------------

    if motorcycle > bicycle:

        return "전동킥보드/오토바이 주행"

    return "이륜 이동수단"

In [4]:
results = []

for path in json_files:

    with open(
        path,
        encoding="utf-8"
    ) as f:

        data = json.load(f)

    summary = data["summary"]

    pred = classify_event(summary)

    results.append({

        "file":
        os.path.basename(path),

        "summary":
        summary,

        "prediction":
        pred
    })

for r in results:

    print("\n----------------")

    print(r["file"])

    print(r["summary"])

    print("예측:", r["prediction"])


----------------
bs_e4387_c1.json
{'person': 34, 'bicycle': 24}
예측: 자전거 인도 주행

----------------
bs_e4387_c2.json
{'person': 30, 'bicycle': 7}
예측: 자전거 인도 주행

----------------
es_e3768_c1.json
{'person': 38, 'bicycle': 2, 'motorcycle': 19}
예측: 전동킥보드/오토바이 주행

----------------
es_e3768_c2.json
{'person': 240, 'bicycle': 8, 'motorcycle': 25}
예측: 이륜 이동수단 헬멧 미착용 후보

----------------
md_e2122_c1.json
{'person': 348, 'motorcycle': 91}
예측: 이륜 이동수단 헬멧 미착용 후보

----------------
md_e2122_c2.json
{'person': 132, 'motorcycle': 24}
예측: 이륜 이동수단 헬멧 미착용 후보

----------------
mp_e2730_c1.json
{'person': 40, 'motorcycle': 31}
예측: 이륜 이동수단 헬멧 미착용 후보

----------------
mp_e2730_c2.json
{'person': 37, 'motorcycle': 26}
예측: 이륜 이동수단 헬멧 미착용 후보

----------------
ms_e3500_c1.json
{'person': 10}
예측: 기타

----------------
ms_e3500_c2.json
{}
예측: 기타


In [5]:
import math

def distance(p1, p2):

    return math.sqrt(
        (p1[0]-p2[0])**2 +
        (p1[1]-p2[1])**2
    )

In [11]:
def analyze_frame(frame_info):

    persons = []
    vehicles = []

    for obj in frame_info["objects"]:

        if obj["class"] == "person":
            persons.append(obj)

        elif obj["class"] in [
            "motorcycle",
            "bicycle"
        ]:
            vehicles.append(obj)

    pair_count = 0

    for p in persons:

        for v in vehicles:

            p_center = get_center(p)
            v_center = get_center(v)

            d = distance(
                p_center,
                v_center
            )

            if d < 150:

                pair_count += 1

    return {

        "person_count": len(persons),

        "vehicle_count": len(vehicles),

        "pair_count": pair_count
    }

In [10]:
def get_center(obj):

    x1,y1,x2,y2 = obj["bbox"]

    cx = (x1+x2)/2
    cy = (y1+y2)/2

    return (cx,cy)

In [7]:
def analyze_video(data):

    riding_frames = 0

    vehicle_frames = 0

    person_frames = 0

    for frame in data["frames"]:

        result = analyze_frame(frame)

        if result["vehicle_count"] > 0:
            vehicle_frames += 1

        if result["person_count"] > 0:
            person_frames += 1

        if result["pair_count"] > 0:
            riding_frames += 1

    return {

        "vehicle_frames":
        vehicle_frames,

        "person_frames":
        person_frames,

        "riding_frames":
        riding_frames
    }

In [8]:
import json

sample_file = os.path.join(
    JSON_DIR,
    "md_e2122_c1.json"
)

with open(
    sample_file,
    encoding="utf-8"
) as f:

    data = json.load(f)

feature = analyze_video(data)

print(feature)

KeyError: 'center'

In [9]:
import json

with open(json_files[0], encoding="utf-8") as f:
    data = json.load(f)

print(
    json.dumps(
        data["frames"][0],
        indent=4,
        ensure_ascii=False
    )
)

{
    "frame_id": 0,
    "classes": [
        "person"
    ],
    "object_count": 1,
    "objects": [
        {
            "class": "person",
            "confidence": 0.5783,
            "bbox": [
                580,
                120,
                614,
                231
            ]
        }
    ]
}


In [12]:
feature = analyze_video(data)

print(feature)

{'vehicle_frames': 18, 'person_frames': 30, 'riding_frames': 10}


In [13]:
for path in json_files:

    with open(path, encoding="utf-8") as f:
        data = json.load(f)

    feature = analyze_video(data)

    ratio = 0

    if feature["vehicle_frames"] > 0:

        ratio = (
            feature["riding_frames"]
            /
            feature["vehicle_frames"]
        )

    print("\n----------------")

    print(os.path.basename(path))

    print(feature)

    print(
        f"riding_ratio={ratio:.2f}"
    )


----------------
bs_e4387_c1.json
{'vehicle_frames': 18, 'person_frames': 30, 'riding_frames': 10}
riding_ratio=0.56

----------------
bs_e4387_c2.json
{'vehicle_frames': 4, 'person_frames': 21, 'riding_frames': 4}
riding_ratio=1.00

----------------
es_e3768_c1.json
{'vehicle_frames': 17, 'person_frames': 32, 'riding_frames': 4}
riding_ratio=0.24

----------------
es_e3768_c2.json
{'vehicle_frames': 22, 'person_frames': 92, 'riding_frames': 16}
riding_ratio=0.73

----------------
md_e2122_c1.json
{'vehicle_frames': 58, 'person_frames': 92, 'riding_frames': 31}
riding_ratio=0.53

----------------
md_e2122_c2.json
{'vehicle_frames': 24, 'person_frames': 62, 'riding_frames': 1}
riding_ratio=0.04

----------------
mp_e2730_c1.json
{'vehicle_frames': 29, 'person_frames': 38, 'riding_frames': 25}
riding_ratio=0.86

----------------
mp_e2730_c2.json
{'vehicle_frames': 25, 'person_frames': 29, 'riding_frames': 24}
riding_ratio=0.96

----------------
ms_e3500_c1.json
{'vehicle_frames': 0, 'pe

In [14]:
def analyze_video_v2(data):

    vehicle_frames = 0
    person_frames = 0
    riding_frames = 0

    max_consecutive_vehicle = 0
    current_vehicle = 0

    for frame in data["frames"]:

        result = analyze_frame(frame)

        if result["vehicle_count"] > 0:

            vehicle_frames += 1

            current_vehicle += 1

            max_consecutive_vehicle = max(
                max_consecutive_vehicle,
                current_vehicle
            )

        else:

            current_vehicle = 0

        if result["person_count"] > 0:
            person_frames += 1

        if result["pair_count"] > 0:
            riding_frames += 1

    return {

        "vehicle_frames": vehicle_frames,

        "person_frames": person_frames,

        "riding_frames": riding_frames,

        "max_consecutive_vehicle":
        max_consecutive_vehicle
    }

In [15]:
for path in json_files:

    with open(path, encoding="utf-8") as f:
        data = json.load(f)

    feature = analyze_video_v2(data)

    print("\n----------------")
    print(os.path.basename(path))
    print(feature)


----------------
bs_e4387_c1.json
{'vehicle_frames': 18, 'person_frames': 30, 'riding_frames': 10, 'max_consecutive_vehicle': 10}

----------------
bs_e4387_c2.json
{'vehicle_frames': 4, 'person_frames': 21, 'riding_frames': 4, 'max_consecutive_vehicle': 4}

----------------
es_e3768_c1.json
{'vehicle_frames': 17, 'person_frames': 32, 'riding_frames': 4, 'max_consecutive_vehicle': 7}

----------------
es_e3768_c2.json
{'vehicle_frames': 22, 'person_frames': 92, 'riding_frames': 16, 'max_consecutive_vehicle': 13}

----------------
md_e2122_c1.json
{'vehicle_frames': 58, 'person_frames': 92, 'riding_frames': 31, 'max_consecutive_vehicle': 22}

----------------
md_e2122_c2.json
{'vehicle_frames': 24, 'person_frames': 62, 'riding_frames': 1, 'max_consecutive_vehicle': 7}

----------------
mp_e2730_c1.json
{'vehicle_frames': 29, 'person_frames': 38, 'riding_frames': 25, 'max_consecutive_vehicle': 28}

----------------
mp_e2730_c2.json
{'vehicle_frames': 25, 'person_frames': 29, 'riding_fra

In [16]:
def get_summary_feature(summary):

    person = summary.get("person",0)

    bicycle = summary.get("bicycle",0)

    motorcycle = summary.get("motorcycle",0)

    two_wheel = bicycle + motorcycle

    if two_wheel == 0:

        pm_ratio = 0

    else:

        pm_ratio = (
            person /
            two_wheel
        )

    return {

        "person": person,

        "bicycle": bicycle,

        "motorcycle": motorcycle,

        "two_wheel": two_wheel,

        "person_vehicle_ratio":
        round(pm_ratio,2)
    }

In [17]:
for path in json_files:

    with open(path, encoding="utf-8") as f:
        data = json.load(f)

    print("\n")
    print(os.path.basename(path))

    print(
        get_summary_feature(
            data["summary"]
        )
    )



bs_e4387_c1.json
{'person': 34, 'bicycle': 24, 'motorcycle': 0, 'two_wheel': 24, 'person_vehicle_ratio': 1.42}


bs_e4387_c2.json
{'person': 30, 'bicycle': 7, 'motorcycle': 0, 'two_wheel': 7, 'person_vehicle_ratio': 4.29}


es_e3768_c1.json
{'person': 38, 'bicycle': 2, 'motorcycle': 19, 'two_wheel': 21, 'person_vehicle_ratio': 1.81}


es_e3768_c2.json
{'person': 240, 'bicycle': 8, 'motorcycle': 25, 'two_wheel': 33, 'person_vehicle_ratio': 7.27}


md_e2122_c1.json
{'person': 348, 'bicycle': 0, 'motorcycle': 91, 'two_wheel': 91, 'person_vehicle_ratio': 3.82}


md_e2122_c2.json
{'person': 132, 'bicycle': 0, 'motorcycle': 24, 'two_wheel': 24, 'person_vehicle_ratio': 5.5}


mp_e2730_c1.json
{'person': 40, 'bicycle': 0, 'motorcycle': 31, 'two_wheel': 31, 'person_vehicle_ratio': 1.29}


mp_e2730_c2.json
{'person': 37, 'bicycle': 0, 'motorcycle': 26, 'two_wheel': 26, 'person_vehicle_ratio': 1.42}


ms_e3500_c1.json
{'person': 10, 'bicycle': 0, 'motorcycle': 0, 'two_wheel': 0, 'person_vehicle

In [18]:
def extract_feature(data):

    summary = data["summary"]

    person = summary.get(
        "person",
        0
    )

    bicycle = summary.get(
        "bicycle",
        0
    )

    motorcycle = summary.get(
        "motorcycle",
        0
    )

    two_wheel = (
        bicycle +
        motorcycle
    )

    if two_wheel == 0:

        bicycle_ratio = 0
        motorcycle_ratio = 0
        person_vehicle_ratio = 0

    else:

        bicycle_ratio = (
            bicycle /
            two_wheel
        )

        motorcycle_ratio = (
            motorcycle /
            two_wheel
        )

        person_vehicle_ratio = (
            person /
            two_wheel
        )

    video_feature = analyze_video(data)

    if video_feature["vehicle_frames"] == 0:

        riding_ratio = 0

    else:

        riding_ratio = (
            video_feature["riding_frames"]
            /
            video_feature["vehicle_frames"]
        )

    return {

        "person": person,

        "bicycle": bicycle,

        "motorcycle": motorcycle,

        "two_wheel": two_wheel,

        "bicycle_ratio":
        round(
            bicycle_ratio,
            2
        ),

        "motorcycle_ratio":
        round(
            motorcycle_ratio,
            2
        ),

        "person_vehicle_ratio":
        round(
            person_vehicle_ratio,
            2
        ),

        "riding_ratio":
        round(
            riding_ratio,
            2
        )
    }

In [19]:
for path in json_files:

    with open(path,
              encoding="utf-8") as f:

        data = json.load(f)

    print(
        "\n",
        os.path.basename(path)
    )

    print(
        extract_feature(data)
    )


 bs_e4387_c1.json
{'person': 34, 'bicycle': 24, 'motorcycle': 0, 'two_wheel': 24, 'bicycle_ratio': 1.0, 'motorcycle_ratio': 0.0, 'person_vehicle_ratio': 1.42, 'riding_ratio': 0.56}

 bs_e4387_c2.json
{'person': 30, 'bicycle': 7, 'motorcycle': 0, 'two_wheel': 7, 'bicycle_ratio': 1.0, 'motorcycle_ratio': 0.0, 'person_vehicle_ratio': 4.29, 'riding_ratio': 1.0}

 es_e3768_c1.json
{'person': 38, 'bicycle': 2, 'motorcycle': 19, 'two_wheel': 21, 'bicycle_ratio': 0.1, 'motorcycle_ratio': 0.9, 'person_vehicle_ratio': 1.81, 'riding_ratio': 0.24}

 es_e3768_c2.json
{'person': 240, 'bicycle': 8, 'motorcycle': 25, 'two_wheel': 33, 'bicycle_ratio': 0.24, 'motorcycle_ratio': 0.76, 'person_vehicle_ratio': 7.27, 'riding_ratio': 0.73}

 md_e2122_c1.json
{'person': 348, 'bicycle': 0, 'motorcycle': 91, 'two_wheel': 91, 'bicycle_ratio': 0.0, 'motorcycle_ratio': 1.0, 'person_vehicle_ratio': 3.82, 'riding_ratio': 0.53}

 md_e2122_c2.json
{'person': 132, 'bicycle': 0, 'motorcycle': 24, 'two_wheel': 24, 'bicy

In [20]:
def calc_iou(box1, box2):

    xA = max(box1[0], box2[0])
    yA = max(box1[1], box2[1])

    xB = min(box1[2], box2[2])
    yB = min(box1[3], box2[3])

    inter_w = max(0, xB - xA)
    inter_h = max(0, yB - yA)

    inter_area = inter_w * inter_h

    area1 = (
        (box1[2] - box1[0]) *
        (box1[3] - box1[1])
    )

    area2 = (
        (box2[2] - box2[0]) *
        (box2[3] - box2[1])
    )

    union = area1 + area2 - inter_area

    if union <= 0:
        return 0

    return inter_area / union

In [21]:
def get_center(box):

    x1,y1,x2,y2 = box

    return (
        (x1+x2)/2,
        (y1+y2)/2
    )

In [22]:
def rider_on_vehicle(
    person_box,
    vehicle_box
):

    px, py = get_center(person_box)

    vx1, vy1, vx2, vy2 = vehicle_box

    inside_x = (
        vx1 <= px <= vx2
    )

    inside_y = (
        vy1 <= py <= vy2
    )

    return inside_x and inside_y

In [23]:
def analyze_frame(frame_info):

    persons = []
    vehicles = []

    for obj in frame_info["objects"]:

        cls = obj["class"]

        if cls == "person":

            persons.append(obj)

        elif cls in [
            "bicycle",
            "motorcycle"
        ]:

            vehicles.append(obj)

    riding_pairs = 0

    for p in persons:

        p_box = p["bbox"]

        for v in vehicles:

            v_box = v["bbox"]

            iou = calc_iou(
                p_box,
                v_box
            )

            center_match = rider_on_vehicle(
                p_box,
                v_box
            )

            if (
                iou > 0.01
                or
                center_match
            ):

                riding_pairs += 1

                break

    return {

        "person_count":
        len(persons),

        "vehicle_count":
        len(vehicles),

        "riding_pairs":
        riding_pairs
    }

In [24]:
def analyze_video_v3(data):

    vehicle_frames = 0

    riding_frames = 0

    max_riding_streak = 0

    current_streak = 0

    total_pairs = 0

    for frame in data["frames"]:

        result = analyze_frame(frame)

        if result["vehicle_count"] > 0:

            vehicle_frames += 1

        if result["riding_pairs"] > 0:

            riding_frames += 1

            current_streak += 1

            total_pairs += (
                result["riding_pairs"]
            )

        else:

            current_streak = 0

        max_riding_streak = max(

            max_riding_streak,

            current_streak
        )

    riding_ratio = 0

    if vehicle_frames > 0:

        riding_ratio = (
            riding_frames /
            vehicle_frames
        )

    return {

        "vehicle_frames":
        vehicle_frames,

        "riding_frames":
        riding_frames,

        "riding_ratio":
        round(
            riding_ratio,
            3
        ),

        "total_pairs":
        total_pairs,

        "max_riding_streak":
        max_riding_streak
    }

In [25]:
for path in json_files:

    with open(
        path,
        encoding="utf-8"
    ) as f:

        data = json.load(f)

    feature = analyze_video_v3(data)

    print("\n----------------")

    print(
        os.path.basename(path)
    )

    print(feature)


----------------
bs_e4387_c1.json
{'vehicle_frames': 18, 'riding_frames': 10, 'riding_ratio': 0.556, 'total_pairs': 10, 'max_riding_streak': 2}

----------------
bs_e4387_c2.json
{'vehicle_frames': 4, 'riding_frames': 4, 'riding_ratio': 1.0, 'total_pairs': 4, 'max_riding_streak': 4}

----------------
es_e3768_c1.json
{'vehicle_frames': 17, 'riding_frames': 2, 'riding_ratio': 0.118, 'total_pairs': 2, 'max_riding_streak': 1}

----------------
es_e3768_c2.json
{'vehicle_frames': 22, 'riding_frames': 13, 'riding_ratio': 0.591, 'total_pairs': 14, 'max_riding_streak': 4}

----------------
md_e2122_c1.json
{'vehicle_frames': 58, 'riding_frames': 27, 'riding_ratio': 0.466, 'total_pairs': 34, 'max_riding_streak': 8}

----------------
md_e2122_c2.json
{'vehicle_frames': 24, 'riding_frames': 0, 'riding_ratio': 0.0, 'total_pairs': 0, 'max_riding_streak': 0}

----------------
mp_e2730_c1.json
{'vehicle_frames': 29, 'riding_frames': 25, 'riding_ratio': 0.862, 'total_pairs': 26, 'max_riding_streak':

In [26]:
##행동추출 시작

In [27]:
import math

def distance(p1, p2):

    return math.sqrt(

        (p1[0]-p2[0])**2 +

        (p1[1]-p2[1])**2
    )

In [28]:
def box_center(box):

    x1,y1,x2,y2 = box

    return (

        (x1+x2)/2,

        (y1+y2)/2
    )

In [29]:
def movement_score(data):

    centers = []

    for frame in data["frames"]:

        vehicles = [

            obj

            for obj in frame["objects"]

            if obj["class"]

            in ["motorcycle","bicycle"]

        ]

        if len(vehicles)==0:
            continue

        biggest = max(

            vehicles,

            key=lambda x:

            (x["bbox"][2]-x["bbox"][0]) *

            (x["bbox"][3]-x["bbox"][1])

        )

        centers.append(

            box_center(
                biggest["bbox"]
            )
        )

    score = 0

    for i in range(
        len(centers)-1
    ):

        score += distance(

            centers[i],

            centers[i+1]
        )

    return round(score,2)

In [30]:
for path in json_files:

    with open(
        path,
        encoding="utf-8"
    ) as f:

        data = json.load(f)

    score = movement_score(data)

    print(
        os.path.basename(path),
        score
    )

bs_e4387_c1.json 1037.82
bs_e4387_c2.json 269.04
es_e3768_c1.json 2555.16
es_e3768_c2.json 3788.11
md_e2122_c1.json 4778.15
md_e2122_c2.json 635.55
mp_e2730_c1.json 670.37
mp_e2730_c2.json 831.91
ms_e3500_c1.json 0
ms_e3500_c2.json 0


In [31]:
def classify_situation(feature):

    # 오토바이 계열

    if (
        feature["motorcycle_ratio"] > 0.7
        and
        feature["riding_ratio"] > 0.3
    ):

        return "motorcycle_riding"

    # 자전거 계열

    if (
        feature["bicycle_ratio"] > 0.7
        and
        feature["riding_ratio"] > 0.3
    ):

        return "bicycle_riding"

    return "unknown"

In [32]:
def classify_event(feature):

    situation = classify_situation(feature)

    if situation == "bicycle_riding":

        return "이륜 이동수단 인도 주행"

    if situation == "motorcycle_riding":

        return "이륜 이동수단 헬멧 미착용(후보)"

    return "판정불가"
    

In [34]:
for frame in data["frames"]:

    if len(frame["objects"]) > 0:

        print(
            json.dumps(
                frame,
                indent=4,
                ensure_ascii=False
            )
        )

        break
        

In [36]:
print(type(data))
print(data.keys())
print(len(data["frames"]))

<class 'dict'>
dict_keys(['summary', 'frame_summary', 'frames'])
91


In [37]:
sum(
    len(frame["objects"])
    for frame in data["frames"]
)

0

In [38]:
print(json_files[0])

B_PART_LIGHT\bs_e4387_c1.json


In [39]:
import os

for d in os.listdir("."):
    print(d)

.config
.ipynb_checkpoints
.ipython
.jupyter
.keras
.matplotlib
.node_repl_history
.vscode
.vscode-shared
20223146우성한딥러닝과제3.ipynb
20223146우성한딥러닝과제3.pdf
21.다각도 CCTV 생활안전 데이터
3D Objects
AMDRM_Install.log
ansel
AppData
Application Data
B_PART_LIGHT
check_0.jpg
check_239.jpg
check_478.jpg
check_717.jpg
classifier.ipynb
Contacts
Cookies
Desktop
Documents
Downloads
Favorites
Links
Local Settings
Music
My Documents
NetHood
NTUSER.DAT
ntuser.dat.LOG1
ntuser.dat.LOG2
NTUSER.DAT{2ad838bc-efea-11ee-a54d-000d3a94eaa1}.TM.blf
NTUSER.DAT{2ad838bc-efea-11ee-a54d-000d3a94eaa1}.TMContainer00000000000000000001.regtrans-ms
NTUSER.DAT{2ad838bc-efea-11ee-a54d-000d3a94eaa1}.TMContainer00000000000000000002.regtrans-ms
ntuser.ini
OneDrive
output
output_part2
part1.ipynb
part2.ipynb
Pictures
PrintHood
Recent
Sample (1)
sam_crop
Saved Games
Searches
SendTo
Templates
test
test_crop.jpg
test_frame.jpg
Untitled.ipynb
Videos
yolov8n.pt
yolov8s.pt
시작 메뉴


In [40]:
import os

for root, dirs, files in os.walk("output"):
    for f in files:
        if f.endswith(".json"):
            print(os.path.join(root, f))

output\metadata.json
output\.ipynb_checkpoints\metadata-checkpoint.json
output\detections\bs_e4387_c1.json
output\detections\bs_e4387_c2.json
output\detections\es_e3768_c1.json
output\detections\es_e3768_c2.json
output\detections\md_e2122_c1.json
output\detections\md_e2122_c2.json
output\detections\mp_e2730_c1.json
output\detections\mp_e2730_c2.json
output\detections\ms_e3500_c1.json
output\detections\ms_e3500_c2.json


In [41]:
import os

for root, dirs, files in os.walk("output_part2"):
    for f in files:
        if f.endswith(".json"):
            print(os.path.join(root, f))

output_part2\tracking_result.json
output_part2\.ipynb_checkpoints\tracking_result-checkpoint.json


In [42]:
import json

path = r"output\detections\bs_e4387_c1.json"

with open(path, encoding="utf-8") as f:
    data = json.load(f)

print(data.keys())

dict_keys(['video_name', 'total_frames', 'summary', 'frames'])


In [43]:
print(len(data["frames"]))

451


In [44]:
for frame in data["frames"]:

    if len(frame["objects"]) > 0:

        print(
            json.dumps(
                frame,
                indent=4,
                ensure_ascii=False
            )
        )

        break

{
    "frame_id": 80,
    "object_count": 1,
    "classes": [
        "person"
    ],
    "objects": [
        {
            "class": "person",
            "confidence": 0.2843,
            "bbox": [
                1264,
                680,
                1400,
                1045
            ]
        }
    ]
}


In [45]:
def bbox_center(bbox):

    x1, y1, x2, y2 = bbox

    return (
        (x1 + x2) / 2,
        (y1 + y2) / 2
    )

In [46]:
def analyze_frame(frame):

    persons = []
    vehicles = []

    for obj in frame["objects"]:

        cls = obj["class"]

        if cls == "person":

            persons.append(
                bbox_center(
                    obj["bbox"]
                )
            )

        elif cls in [
            "bicycle",
            "motorcycle"
        ]:

            vehicles.append(
                bbox_center(
                    obj["bbox"]
                )
            )

    pair_count = 0

    for p in persons:

        for v in vehicles:

            dist = (
                (p[0]-v[0])**2 +
                (p[1]-v[1])**2
            ) ** 0.5

            if dist < 150:

                pair_count += 1

    return {

        "person_count":
            len(persons),

        "vehicle_count":
            len(vehicles),

        "pair_count":
            pair_count
    }

In [47]:
for frame in data["frames"]:

    if frame["object_count"] > 0:

        print(
            analyze_frame(frame)
        )

        break

{'person_count': 1, 'vehicle_count': 0, 'pair_count': 0}


In [48]:
def extract_video_feature(data):

    vehicle_frames = 0
    riding_frames = 0

    max_streak = 0
    current_streak = 0

    total_pairs = 0

    for frame in data["frames"]:

        result = analyze_frame(frame)

        if result["vehicle_count"] > 0:

            vehicle_frames += 1

        if result["pair_count"] > 0:

            riding_frames += 1

            total_pairs += result["pair_count"]

            current_streak += 1

            max_streak = max(
                max_streak,
                current_streak
            )

        else:

            current_streak = 0

    riding_ratio = 0

    if vehicle_frames > 0:

        riding_ratio = (
            riding_frames /
            vehicle_frames
        )

    return {

        "vehicle_frames":
            vehicle_frames,

        "riding_frames":
            riding_frames,

        "riding_ratio":
            round(riding_ratio,3),

        "total_pairs":
            total_pairs,

        "max_riding_streak":
            max_streak
    }

In [49]:
for json_file in json_files:

    with open(json_file, encoding="utf-8") as f:

        data = json.load(f)

    print(
        os.path.basename(json_file)
    )

    print(
        extract_video_feature(data)
    )

    print()

bs_e4387_c1.json
{'vehicle_frames': 18, 'riding_frames': 10, 'riding_ratio': 0.556, 'total_pairs': 13, 'max_riding_streak': 2}

bs_e4387_c2.json
{'vehicle_frames': 4, 'riding_frames': 4, 'riding_ratio': 1.0, 'total_pairs': 7, 'max_riding_streak': 4}

es_e3768_c1.json
{'vehicle_frames': 17, 'riding_frames': 4, 'riding_ratio': 0.235, 'total_pairs': 4, 'max_riding_streak': 1}

es_e3768_c2.json
{'vehicle_frames': 22, 'riding_frames': 16, 'riding_ratio': 0.727, 'total_pairs': 29, 'max_riding_streak': 5}

md_e2122_c1.json
{'vehicle_frames': 58, 'riding_frames': 31, 'riding_ratio': 0.534, 'total_pairs': 67, 'max_riding_streak': 11}

md_e2122_c2.json
{'vehicle_frames': 24, 'riding_frames': 1, 'riding_ratio': 0.042, 'total_pairs': 1, 'max_riding_streak': 1}

mp_e2730_c1.json
{'vehicle_frames': 29, 'riding_frames': 25, 'riding_ratio': 0.862, 'total_pairs': 28, 'max_riding_streak': 25}

mp_e2730_c2.json
{'vehicle_frames': 25, 'riding_frames': 24, 'riding_ratio': 0.96, 'total_pairs': 31, 'max_ridi

In [50]:
import math

def center(bbox):

    x1,y1,x2,y2 = bbox

    return (
        (x1+x2)/2,
        (y1+y2)/2
    )

def distance(p1,p2):

    return math.sqrt(
        (p1[0]-p2[0])**2 +
        (p1[1]-p2[1])**2
    )

In [51]:
import math

def center(bbox):

    x1,y1,x2,y2 = bbox

    return (
        (x1+x2)/2,
        (y1+y2)/2
    )

def distance(p1,p2):

    return math.sqrt(
        (p1[0]-p2[0])**2 +
        (p1[1]-p2[1])**2
    )

In [53]:
def estimate_motion(data):

    prev_vehicle = None

    movement = []

    for frame in data["frames"]:

        vehicles = []

        for obj in frame["objects"]:

            if obj["class"] in [
                "bicycle",
                "motorcycle"
            ]:

                vehicles.append(
                    center(
                        obj["bbox"]
                    )
                )

        if len(vehicles)==0:

            continue

        current = vehicles[0]

        if prev_vehicle is not None:

            move = distance(
                current,
                prev_vehicle
            )

            movement.append(move)

        prev_vehicle = current

    return movement

In [54]:
movement = estimate_motion(data)

print(
    len(movement)
)

print(
    movement[:20]
)

0
[]


In [55]:
vehicle_count = 0

for frame in data["frames"]:

    for obj in frame["objects"]:

        if obj["class"] in [
            "bicycle",
            "motorcycle"
        ]:

            vehicle_count += 1

print(vehicle_count)

0


In [56]:
for frame in data["frames"]:

    if frame["object_count"] > 0:

        print(frame["frame_id"])

        break